In [ ]:
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, top_k_accuracy_score
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression
from google.colab import files


# =========================================================
# 1. Load dataset
# =========================================================
travel = pd.read_csv("/content/travelling_dataset.csv")

print("Dataset shape:", travel.shape)
print(travel.head())


# =========================================================
# 2. Clean text columns
# =========================================================
travel["country"] = travel["country"].str.strip()
travel["budget"] = travel["budget"].str.strip().str.lower()
travel["climate"] = travel["climate"].str.strip().str.lower()
travel["activities"] = travel["activities"].str.strip().str.lower()


# =========================================================
# 3. Split features and target
# =========================================================
X = travel.drop("country", axis=1)
y = travel["country"]


# =========================================================
# 4. Encode features
# =========================================================
numeric_features = X[[
    "safety",
    "food",
    "nightlife",
    "nature",
    "family_friendly"
]]

budget_encoded = pd.get_dummies(X["budget"], prefix="budget", dtype=int)
climate_encoded = pd.get_dummies(X["climate"], prefix="climate", dtype=int)

activities_encoded = X["activities"].str.get_dummies(sep=", ")
activities_encoded = activities_encoded.add_prefix("activity_")

X_encoded = pd.concat(
    [
        numeric_features,
        budget_encoded,
        climate_encoded,
        activities_encoded
    ],
    axis=1
)

print("Encoded shape:", X_encoded.shape)
print("Duplicate columns:", X_encoded.columns.duplicated().sum())


# =========================================================
# 5. Encode target countries
# =========================================================
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

print("Number of countries:", len(label_encoder.classes_))


# =========================================================
# 6. Train-test split
# =========================================================
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded,
    y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)


# =========================================================
# 7. Train several models and choose best
# =========================================================
models = {
    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        class_weight="balanced"
    ),
    "Extra Trees": ExtraTreesClassifier(
        n_estimators=300,
        random_state=42,
        class_weight="balanced"
    ),
    "Logistic Regression": LogisticRegression(
        max_iter=5000,
        class_weight="balanced"
    )
}

best_model = None
best_accuracy = 0
best_name = ""

for name, model in models.items():
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)

    print("=" * 50)
    print(name)
    print("Accuracy:", accuracy)

    if accuracy > best_accuracy:
        best_accuracy = accuracy
        best_model = model
        best_name = name


print("=" * 50)
print("Best model:", best_name)
print("Best accuracy:", best_accuracy)


# =========================================================
# 8. Evaluate best model
# =========================================================
y_pred = best_model.predict(X_test)

print("\nClassification report:")
print(classification_report(
    y_test,
    y_pred,
    target_names=label_encoder.classes_,
    zero_division=0
))

if hasattr(best_model, "predict_proba"):
    y_proba = best_model.predict_proba(X_test)

    top5_accuracy = top_k_accuracy_score(
        y_test,
        y_proba,
        k=5,
        labels=list(range(len(label_encoder.classes_)))
    )

    print("Top-5 Accuracy:", top5_accuracy)


# =========================================================
# 9. Save model files
# =========================================================
joblib.dump(best_model, "travel_country_model.pkl")
joblib.dump(label_encoder, "label_encoder.pkl")
joblib.dump(X_encoded.columns, "model_columns.pkl")

print("\nModel saved successfully!")
print("Saved files:")
print("1. travel_country_model.pkl")
print("2. label_encoder.pkl")
print("3. model_columns.pkl")


# =========================================================
# 10. Download model files from Colab
# =========================================================
files.download("travel_country_model.pkl")
files.download("label_encoder.pkl")
files.download("model_columns.pkl")


# =========================================================
# 11. Load saved model files
# Use this later when you want to use the saved model again
# =========================================================
loaded_model = joblib.load("travel_country_model.pkl")
loaded_label_encoder = joblib.load("label_encoder.pkl")
loaded_model_columns = joblib.load("model_columns.pkl")


# =========================================================
# 12. Function to predict TOP 5 countries
# =========================================================
def predict_top_5_countries(
    safety,
    food,
    nightlife,
    nature,
    family_friendly,
    budget,
    climate,
    activities
):
    new_traveler = pd.DataFrame([{
        "safety": safety,
        "food": food,
        "nightlife": nightlife,
        "nature": nature,
        "family_friendly": family_friendly,
        "budget": budget,
        "climate": climate,
        "activities": activities
    }])

    new_traveler["budget"] = new_traveler["budget"].str.strip().str.lower()
    new_traveler["climate"] = new_traveler["climate"].str.strip().str.lower()
    new_traveler["activities"] = new_traveler["activities"].str.strip().str.lower()

    new_numeric = new_traveler[[
        "safety",
        "food",
        "nightlife",
        "nature",
        "family_friendly"
    ]]

    new_budget = pd.get_dummies(
        new_traveler["budget"],
        prefix="budget",
        dtype=int
    )

    new_climate = pd.get_dummies(
        new_traveler["climate"],
        prefix="climate",
        dtype=int
    )

    new_activities = new_traveler["activities"].str.get_dummies(sep=", ")
    new_activities = new_activities.add_prefix("activity_")

    new_encoded = pd.concat(
        [
            new_numeric,
            new_budget,
            new_climate,
            new_activities
        ],
        axis=1
    )

    new_encoded = new_encoded.reindex(
        columns=loaded_model_columns,
        fill_value=0
    )

    probabilities = loaded_model.predict_proba(new_encoded)[0]

    top_5_indexes = probabilities.argsort()[-5:][::-1]
    top_5_countries = loaded_label_encoder.inverse_transform(top_5_indexes)
    top_5_probabilities = probabilities[top_5_indexes] * 100

    results = pd.DataFrame({
        "Rank": [1, 2, 3, 4, 5],
        "Country": top_5_countries,
        "Probability (%)": top_5_probabilities.round(2)
    })

    return results


# =========================================================
# 13. Test prediction
# =========================================================
result = predict_top_5_countries(
    safety=5,
    food=4,
    nightlife=3,
    nature=5,
    family_friendly=4,
    budget="medium",
    climate="warm",
    activities="beaches, nature"
)

print("\nTop 5 predicted countries:")
print(result)

Dataset shape: (1224, 9)
         country  budget   climate                  activities  safety  food  \
0  United States  medium     mixed         city, museums, food       5     4   
1  United States  medium     mixed  nature, hiking, road trips       5     5   
2  United States  medium     mixed         city, museums, food       3     3   
3  United States  medium      warm  nature, hiking, road trips       5     5   
4  United States    high  tropical         city, museums, food       3     5   

   nightlife  nature  family_friendly  
0          3       3                3  
1          3       5                4  
2          5       5                3  
3          5       4                3  
4          4       4                4  
Encoded shape: (1224, 46)
Duplicate columns: 0
Number of countries: 102
Random Forest
Accuracy: 0.19591836734693877
Extra Trees
Accuracy: 0.2
Logistic Regression
Accuracy: 0.18775510204081633
Best model: Extra Trees
Best accuracy: 0.2

Classification rep

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Top 5 predicted countries:
   Rank           Country  Probability (%)
0     1            Brazil            18.67
1     2           Vanuatu            16.67
2     3  Papua New Guinea            12.00
3     4          Colombia             5.00
4     5         Australia             4.67
